# AMPS I · Week 04 — Lab
## Functions, interpolation, and measuring errors

| | |
|---|---|
| **Lecture sections** | Part B (`fem.Function`, interpolation) · Part C (errors, rates, I/O) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | look inside `fem.Function` in parallel · interpolate formulas, functions, expressions and piecewise data · test polynomial preservation · measure $L^2$/$H^1$ errors and rates · avoid three classic mistakes · write fields for ParaView |

**Reminder.** `%%fenicsx` cells run in a separate MPI process and are the only cells where `basix`, `ufl`, `dolfinx`
are available. Ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · Inside `fem.Function`  *(lecture §B1)*
On 2 ranks: the array holds owned values first, then ghosts. Writing into it yourself requires `scatter_forward()`.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
from dolfinx import fem, mesh

comm = MPI.COMM_WORLD
msh = mesh.create_unit_square(comm, 4, 4)
V = fem.functionspace(msh, ("Lagrange", 2))
u = fem.Function(V, name="u")
im = V.dofmap.index_map
n_own = im.size_local
print(f"rank {comm.rank}: len(u.x.array) = {u.x.array.size} = {n_own} owned + {im.num_ghosts} ghosts", flush=True)

u.x.array[:n_own] = comm.rank + 1                     # each rank writes its OWNED values only
before = np.unique(u.x.array[n_own:]).tolist()
u.x.scatter_forward()                                 # ghosts receive the owners' values
after = np.unique(u.x.array[n_own:]).tolist()
print(f"rank {comm.rank}: ghost values before scatter {before}, after {after}", flush=True)

## 2 · Ways to interpolate  *(lecture §B2–B3)*

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh

msh = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8)
x = ufl.SpatialCoordinate(msh)

# (a) a vectorised NumPy callable — scalar returns (n,), vector returns (gdim, n)
V3 = fem.functionspace(msh, ("Lagrange", 3))
u3 = fem.Function(V3)
u3.interpolate(lambda x: np.sin(np.pi * x[0]) * np.sin(np.pi * x[1]))
Vv = fem.functionspace(msh, ("Lagrange", 1, (2,)))
w = fem.Function(Vv)
w.interpolate(lambda x: np.vstack((-x[1], x[0])))

# (b) three classic mistakes, caught and printed
V1 = fem.functionspace(msh, ("Lagrange", 1))
s = fem.Function(V1)
for label, f in (("constant", lambda x: 1.0), ("transposed vector", None)):
    try:
        if f is None:
            w.interpolate(lambda x: np.stack((-x[1], x[0]), axis=1))
        else:
            s.interpolate(f)
    except Exception as e:
        print(f"{label:18s}: {type(e).__name__}: {str(e).splitlines()[0][:70]}")
try:
    np.sin(x[0])
except Exception as e:
    print(f"{'np.sin on UFL':18s}: {type(e).__name__}: {str(e)[:70]}")

# (c) Function → Function on the same mesh
u1 = fem.Function(V1); u1.interpolate(u3)
d1 = fem.Function(V1); d1.interpolate(lambda x: np.sin(np.pi * x[0]) * np.sin(np.pi * x[1]))
print("P3 → P1 vs direct P1 interpolation:", np.abs(u1.x.array - d1.x.array).max())

# (d) a UFL expression: the gradient of u3 into a DG vector space
VD = fem.functionspace(msh, ("DG", 2, (2,)))
g = fem.Function(VD)
g.interpolate(fem.Expression(ufl.grad(u3), VD.element.interpolation_points))
print("number of DOFs of the DG2 gradient:", g.x.array.size)

# (e) piecewise data on a subset of cells
Q = fem.functionspace(msh, ("DG", 0))
kappa = fem.Function(Q)
left = mesh.locate_entities(msh, 2, lambda x: x[0] <= 0.5 + 1e-12)
right = mesh.locate_entities(msh, 2, lambda x: x[0] >= 0.5 - 1e-12)
kappa.interpolate(lambda x: np.full(x.shape[1], 1.0), cells0=left)
kappa.interpolate(lambda x: np.full(x.shape[1], 10.0), cells0=right)
print("integral of kappa:", fem.assemble_scalar(fem.form(kappa * ufl.dx)))

> **✏️ Your turn (5 min).** Interpolate $u=x^{3/4}$ into P3 with `lambda x: x[0]**0.75`. How many entries of `u.x.array` are `nan`?
> Print the smallest $x$-coordinate of the DOFs (`V.tabulate_dof_coordinates()`), explain, and fix the callable.

In [ ]:
%%fenicsx -np 1
# TODO: count nan values, find the smallest DOF x-coordinate, fix the callable

## 3 · Polynomial preservation — the unit test  *(lecture §A1, §C2)*
$\mathscr I_h p=p$ whenever $p\in\mathscr P_k$. If your pipeline fails this, nothing else it prints can be trusted.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh

def norms(msh, e, q=None):
    """(L2 norm, H1 seminorm) of a UFL expression e, summed over all ranks."""
    md = {} if q is None else {"quadrature_degree": q}
    dx = ufl.Measure("dx", domain=msh, metadata=md)
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * dx)), op=MPI.SUM))
    return s(ufl.inner(e, e)), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

msh = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8)
x = ufl.SpatialCoordinate(msh)
for k in (1, 2, 3):
    u = fem.Function(fem.functionspace(msh, ("Lagrange", k)))
    u.interpolate(lambda x: x[0] ** 2 + x[0] * x[1])
    L2, H1 = norms(msh, u - (x[0] ** 2 + x[0] * x[1]))
    print(f"P{k}: u = x^2 + xy   L2 error {L2:.1e}   H1 error {H1:.1e}")

> **✏️ Your turn (3 min).** Which is the lowest degree $k$ that reproduces $x^3-xy^2$ exactly? And $x^2y^2$? Check.

In [ ]:
%%fenicsx -np 1
# TODO: test k = 1..4 for x^3 - x y^2 and x^2 y^2

## 4 · Errors and rates  *(lecture §C1)*
$u=\sin(2\pi x)\cos(2\pi y)$, P1 and P2, $N=4,\dots,32$. The exact $u$ is a **UFL expression**, not a `Function`.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh

def norms(msh, e, q=None):
    """(L2 norm, H1 seminorm) of a UFL expression e, summed over all ranks."""
    md = {} if q is None else {"quadrature_degree": q}
    dx = ufl.Measure("dx", domain=msh, metadata=md)
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * dx)), op=MPI.SUM))
    return s(ufl.inner(e, e)), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

Ns = [4, 8, 16, 32]
res = {}
for k in (1, 2):
    E = []
    for N in Ns:
        msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
        x = ufl.SpatialCoordinate(msh)
        u = fem.Function(fem.functionspace(msh, ("Lagrange", k)))
        u.interpolate(lambda x: np.sin(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1]))
        E.append(norms(msh, u - ufl.sin(2 * ufl.pi * x[0]) * ufl.cos(2 * ufl.pi * x[1])))
    E = np.array(E)
    res[f"P{k}"] = E
    r = np.log2(E[:-1] / E[1:])
    print(f"P{k}: L2 {E[:, 0].round(6).tolist()} rates {r[:, 0].round(2).tolist()}")
    print(f"    H1 {E[:, 1].round(4).tolist()} rates {r[:, 1].round(2).tolist()}")
np.savez("/tmp/fnx/w04_rates.npz", N=Ns, **res)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
d = np.load("/tmp/fnx/w04_rates.npz")
h = 1 / d["N"]
fig, ax = plt.subplots(figsize=(6, 4))
for k, c in (("P1", "C0"), ("P2", "C1")):
    ax.loglog(h, d[k][:, 0], "o-", color=c, label=f"{k}  L2")
    ax.loglog(h, d[k][:, 1], "s--", color=c, label=f"{k}  H1")
ax.set_xticks(h, [f"1/{n}" for n in d["N"]]); ax.set_xticks([], minor=True)
ax.set_xlabel("h"); ax.legend(); ax.grid(alpha=0.3, which="both"); plt.show()

> **✏️ Your turn (7 min).** Two ways to get a wrong error, on the $N=8$ P1 case:
> (i) interpolate the exact $u$ into the **same** P1 space and use that `Function` as the reference;
> (ii) keep the UFL reference but force `quadrature_degree` = 1, 2, 4, 8. Report the numbers and say which ones you would trust.

In [ ]:
%%fenicsx -np 1
# TODO: (i) reference interpolated into the same space; (ii) quadrature degree 1, 2, 4, 8

## 5 · [D] Interpolation and I/O  *(lecture §C3)*
From the DOLFINx demo: a field whose normal component jumps across $x=0.5$, in an $H(\mathrm{curl})$ space,
then interpolated into discontinuous Lagrange for output. Files go to `/tmp/fnx/out/` (download them for ParaView).

In [ ]:
%%fenicsx -np 1
from pathlib import Path
from mpi4py import MPI
import numpy as np
from dolfinx import fem, geometry, io, mesh
import dolfinx

out = Path("/tmp/fnx/out"); out.mkdir(parents=True, exist_ok=True)
msh = mesh.create_rectangle(MPI.COMM_WORLD, ((0.0, 0.0), (1.0, 1.0)), (16, 16), mesh.CellType.triangle)
V = fem.functionspace(msh, ("N1curl", 1))
u = fem.Function(V)
cells0 = mesh.locate_entities(msh, 2, lambda x: x[0] <= 0.5)
cells1 = mesh.locate_entities(msh, 2, lambda x: x[0] >= 0.5)
u.interpolate(lambda x: np.vstack((x[0], x[1])), cells0)
u.interpolate(lambda x: np.vstack((x[0] + 1, x[1])), cells1)
V0 = fem.functionspace(msh, ("DG", 1, (2,)))
u0 = fem.Function(V0, name="u")
u0.interpolate(u)

# evaluate just left and right of (0.5, 0.3)
pts = np.array([[0.5 - 1e-9, 0.3, 0.0], [0.5 + 1e-9, 0.3, 0.0]])
tree = geometry.bb_tree(msh, msh.topology.dim)
hits = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(tree, pts), pts)
for i, side in enumerate(("left ", "right")):
    print(side, "u0 =", u0.eval(pts[i:i + 1], hits.links(i)[:1]).round(4))

if dolfinx.has_adios2:
    with io.VTXWriter(msh.comm, out / "nedelec.bp", [u0]) as f:
        f.write(0.0)
V2 = fem.functionspace(msh, ("Lagrange", 2))
q = fem.Function(V2, name="q"); q.interpolate(lambda x: np.sin(np.pi * x[0]) * x[1])
try:
    with io.XDMFFile(msh.comm, out / "q2.xdmf", "w") as f:
        f.write_mesh(msh); f.write_function(q)
except RuntimeError as e:
    print("XDMF with P2:", e)
q1 = fem.Function(fem.functionspace(msh, ("Lagrange", 1)), name="q"); q1.interpolate(q)
with io.XDMFFile(msh.comm, out / "q1.xdmf", "w") as f:
    f.write_mesh(msh); f.write_function(q1)
print("has_adios2:", dolfinx.has_adios2, "| files:", sorted(p.name for p in out.iterdir()))

## 6 · LLM-assisted coding: live demo  *(lecture Part D)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11 code for a %%fenicsx -np 2 cell. For u(x, y) = exp(x) * cos(3y) on the unit square,
interpolate u into P2 on N×N meshes with N = 4, 8, 16, 32, compute the L2 norm and the H1 seminorm of
u - I_h u (exact u as a UFL expression), print a table with the convergence rates on rank 0, and save the
errors to /tmp/fnx/llm_rates.npz.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] Rates approach 3 ($L^2$) and 2 ($H^1$) and are steady over the last two refinements
- [ ] Replacing $u$ by $x^2+xy$ gives errors $\sim10^{-15}$ (polynomial preservation)
- [ ] The reference is a UFL expression (an error of exactly 0 means it is not)
- [ ] The numbers are identical with `-np 1` and `-np 2`, and printed once